<a href="https://colab.research.google.com/github/Manidweep27/Benchmarking-Large-Language-Models-for-Materials-Property-Prediction/blob/main/LLM4MAT_PROPERTY_PREDICTION_testmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [16]:
# 1. Clone the repository
!git clone https://github.com/vertaix/LLM4Mat-Bench.git 2>/dev/null || echo "Repository already exists."

# 2. Install required packages
!pip install -q transformers pymatgen accelerate datasets torchmetrics

import os
import pandas as pd

# 3. Create structured evaluation directory mimicking real benchmark structure (e.g. data_path/dataset_name/)
# The script expects 'train.csv', 'validation.csv', and 'test.csv' inside the target dataset folder
os.makedirs("data/mp", exist_ok=True)

# We provide both 'descr' and 'description' to satisfy both the input parameters and the hardcoded prints.
# We must also provide at least 4 items so that hardcoded print statements like test_data['description'][3] succeed.
data = {
    "material_id": ["mp-1", "mp-2", "mp-3", "mp-4", "mp-5"],
    "comp": ["Fe2O3", "LiCoO2", "TiO2", "NaCl", "SiO2"],
    "descr": [
        "A rhombohedral crystal structure with iron atoms octahedrally coordinated by oxygen.",
        "A layered transition metal oxide consisting of lithium and cobalt layers separated by octahedrally coordinated oxygen ions.",
        "Titanium dioxide occurring in nature as well-known minerals rutile, anatase and brookite.",
        "An ionic compound with a 1:1 ratio of sodium and chloride ions.",
        "An oxide of silicon with the chemical formula SiO2, most commonly found in nature as quartz."
    ],
    "description": [
        "A rhombohedral crystal structure with iron atoms octahedrally coordinated by oxygen.",
        "A layered transition metal oxide consisting of lithium and cobalt layers separated by octahedrally coordinated oxygen ions.",
        "Titanium dioxide occurring in nature as well-known minerals rutile, anatase and brookite.",
        "An ionic compound with a 1:1 ratio of sodium and chloride ions.",
        "An oxide of silicon with the chemical formula SiO2, most commonly found in nature as quartz."
    ],
    "band_gap_target": [2.1, 1.4, 3.2, 8.5, 9.0]
}
df = pd.DataFrame(data)

# Save under expected filenames
df.to_csv("data/mp/train.csv", index=False)
df.to_csv("data/mp/validation.csv", index=False)
df.to_csv("data/mp/test.csv", index=False)
print("Generated structured evaluation files with 5 records under data/mp/")

Repository already exists.
Generated structured evaluation files with 5 records under data/mp/


In [3]:
import sys
sys.path.append('/content/LLM4Mat-Bench')

# Let's inspect the repository's evaluation scripts or provide a way to trigger them natively
import os
if os.path.exists('/content/LLM4Mat-Bench/code/llmprop_and_matbert'):
    print("LLM4Mat-Bench directory structure verified.")
else:
    print("Directory structure may differ. Listing repository contents:")
    !find /content/LLM4Mat-Bench -maxdepth 3 -not -path '*/.*'

LLM4Mat-Bench directory structure verified.


In [24]:
import os

repo_root = "/content/LLM4Mat-Bench"
eval_script_path = os.path.join(repo_root, "code/llmprop_and_matbert/evaluate.py")
train_script_path = os.path.join(repo_root, "code/llmprop_and_matbert/train.py")
utils_path = os.path.join(repo_root, "code/llmprop_and_matbert/utils.py")

# Helper to replace deprecated imports, hot-fix tokenizer paths, and update scikit-learn metrics
def apply_hotfixes(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            code_content = f.read()

        # Replace deprecated import
        code_content = code_content.replace(
            "from transformers import AdamW",
            "from torch.optim import AdamW"
        )

        # Redirect missing local tokenizers to use public Hugging Face equivalents
        code_content = code_content.replace(
            'tokenizer = AutoTokenizer.from_pretrained("/content/LLM4Mat-Bench/code/llmprop_and_matbert/tokenizers/llmprop_tokenizer")',
            'tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")'
        )
        code_content = code_content.replace(
            'tokenizer = AutoTokenizer.from_pretrained(f"{tokenizers_path}/llmprop_tokenizer")',
            'tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")'
        )
        code_content = code_content.replace(
            'tokenizer = BertTokenizerFast.from_pretrained(f"{tokenizers_path}/matbert-base-uncased", do_lower_case=True)',
            'tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased", do_lower_case=True)'
        )

        # Hot-fix scikit-learn's deprecated squared parameter in MSE
        code_content = code_content.replace(
            "rmse = metrics.mean_squared_error(targets_list, predictions_list, squared=False)",
            "rmse = np.sqrt(metrics.mean_squared_error(targets_list, predictions_list))"
        )

        with open(file_path, "w") as f:
            f.write(code_content)
        print(f"Successfully hot-fixed {os.path.basename(file_path)}")

apply_hotfixes(eval_script_path)
apply_hotfixes(train_script_path)

# Hot-fix utils.py missing numpy import
if os.path.exists(utils_path):
    with open(utils_path, "r") as f:
        utils_content = f.read()
    if "import numpy as np" not in utils_content:
        utils_content = "import numpy as np\n" + utils_content
        with open(utils_path, "w") as f:
            f.write(utils_content)
        print("Successfully hot-fixed utils.py (added numpy import)")

# 1. Run quick training to generate the required checkpoints (skipping if checkpoints are already generated)
print("\n=== Training Composition Model (1 Epoch) ===")
!python {train_script_path} --input_type comp --data_path /content/data --dataset_name mp --property band_gap_target --epochs 1

print("\n=== Training Description Model (1 Epoch) ===")
!python {train_script_path} --input_type descr --data_path /content/data --dataset_name mp --property band_gap_target --epochs 1

# 2. Run the evaluation scripts natively
print("\n=== Starting Composition Evaluation ===")
!python {eval_script_path} --input_type comp --data_path /content/data --dataset_name mp --property band_gap_target --epochs 1

print("\n=== Starting Description Evaluation ===")
!python {eval_script_path} --input_type descr --data_path /content/data --dataset_name mp --property band_gap_target --epochs 1

Successfully hot-fixed evaluate.py
Successfully hot-fixed train.py
Successfully hot-fixed utils.py (added numpy import)

=== Training Composition Model (1 Epoch) ===
No GPU available, please connect to the GPU first or continue to use CPU instead
--------------------------------------------------
A rhombohedral crystal structure with iron atoms octahedrally coordinated by oxygen.
--------------------------------------------------
An ionic compound with a 1:1 ratio of sodium and chloride ions.
max length: 888
--------------------------------------------------
train data = 5 samples
valid data = 5 samples
test data = 5 samples
--------------------------------------------------
training on 100.0% samples with whole sequence
validating on 100.0% samples with whole sequence
testing on 100.0% samples with whole sequence
--------------------------------------------------
labels statistics on training set:
Mean: tensor(4.8400, dtype=torch.float64)
Standard deviation: tensor(3.6308, dtype=torch